<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/Spark_SQL_Find_Your_Perfect_Weather_sol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🌍 Finding Your Perfect Weather with Spark

In this notebook, you will build a simple data system to answer a real question:

> **Which city has the best weather based on your preferences?**

Different people want different things:

- ☀️ warm and sunny  
- 🪁 windy  
- 🌧️ little or no rain  

Your goal is to **use data to recommend the best city**.

## Your mission

By the end of this notebook, you will be able to:

- find warm and sunny days  
- compare cities based on weather  
- define your own “perfect weather”  

👉 and use Spark to find the best match.

## Getting Started

We begin by collecting real weather data from an API.

An API (Application Programming Interface) allows us to request data from an external service and receive it in **JSON format**.

👉 Below, we define a function `get_weather` that:
- connects to the API  
- requests weather data  
- returns the result for a given city (using latitude and longitude)

In [109]:
import requests

def get_weather(city, latitude, longitude):
    # API endpoint (where we send the request)
    url = "https://api.open-meteo.com/v1/forecast"

    # Parameters sent to the API
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_probability_max",
            "windspeed_10m_max"
        ],
        "timezone": "auto",
        "forecast_days": 16,
        "past_days": 92
    }

    # Send the request to the API
    response = requests.get(url, params=params)

    # Convert the response to JSON format (Python dictionary)
    data = response.json()

    # Add the city name to the data (for later use in Spark)
    data["city"] = city

    # Return the result
    return data

### Get Data for One City

Let’s call the function for one city and store the result.

In [110]:
data = get_weather("Madrid", 40.41, -3.7)

### Inspect the Result

Let’s look at the data returned by the API.

👉 In a notebook, you can display the content of a variable by simply writing its name and running the cell.

In [111]:
data

{'latitude': 40.4375,
 'longitude': -3.6875,
 'generationtime_ms': 0.7630586624145508,
 'utc_offset_seconds': 7200,
 'timezone': 'Europe/Madrid',
 'timezone_abbreviation': 'GMT+2',
 'elevation': 661.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_probability_max': '%',
  'windspeed_10m_max': 'km/h'},
 'daily': {'time': ['2026-01-10',
   '2026-01-11',
   '2026-01-12',
   '2026-01-13',
   '2026-01-14',
   '2026-01-15',
   '2026-01-16',
   '2026-01-17',
   '2026-01-18',
   '2026-01-19',
   '2026-01-20',
   '2026-01-21',
   '2026-01-22',
   '2026-01-23',
   '2026-01-24',
   '2026-01-25',
   '2026-01-26',
   '2026-01-27',
   '2026-01-28',
   '2026-01-29',
   '2026-01-30',
   '2026-01-31',
   '2026-02-01',
   '2026-02-02',
   '2026-02-03',
   '2026-02-04',
   '2026-02-05',
   '2026-02-06',
   '2026-02-07',
   '2026-02-08',
   '2026-02-09',
   '2026-02-10',
   '2026-02-11',
   '2026-02-12',
   '2026-02-13',
   '2026-02-14',
 

# ⚡ Starting Spark

Before working with the data, we need to start a Spark session.

This is the entry point for working with Spark.

In [112]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Spark SQL Intro") \
    .getOrCreate()

## Loading the Data into Spark

We received the data in JSON format.

Now, we will load it into Spark so we can explore its structure and work with it more easily.

In [113]:
import json
df = spark.read.json(
    spark.sparkContext.parallelize([json.dumps(data)])
)

> **Note**
>
> The data we received from the API is a Python object (a dictionary).
>
>Spark cannot work directly with Python variables, so we need to convert it into a format Spark understands.
>
>👉 In simple terms:
>- `json.dumps(...)` converts the data to text  
>- `parallelize(...)` sends it to Spark  
>- `read.json(...)` turns it into a table  
>
>You don’t need to fully understand every part of this yet.

## Inspecting the Data

Now that the data is loaded into Spark, let’s explore its structure.

Spark represents data using a **schema**, which describes how the data is organized.

In [114]:
# ✍️ Exercise:
# Show the schema of the DataFrame we created above
# write your code below:

df.printSchema()

root
 |-- city: string (nullable = true)
 |-- daily: struct (nullable = true)
 |    |-- precipitation_probability_max: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- temperature_2m_max: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_min: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- windspeed_10m_max: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- daily_units: struct (nullable = true)
 |    |-- precipitation_probability_max: string (nullable = true)
 |    |-- temperature_2m_max: string (nullable = true)
 |    |-- temperature_2m_min: string (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- windspeed_10m_max: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullab

In [115]:
# ✍️ Exercise:
# Display the content of the DataFrame
# write your code below:

df.show()

+------+--------------------+--------------------+---------+------------------+--------+---------+-------------+---------------------+------------------+
|  city|               daily|         daily_units|elevation| generationtime_ms|latitude|longitude|     timezone|timezone_abbreviation|utc_offset_seconds|
+------+--------------------+--------------------+---------+------------------+--------+---------+-------------+---------------------+------------------+
|Madrid|{[0, 0, 0, 90, 70...|{%, °C, °C, iso86...|    661.0|0.7630586624145508| 40.4375|  -3.6875|Europe/Madrid|                GMT+2|              7200|
+------+--------------------+--------------------+---------+------------------+--------+---------+-------------+---------------------+------------------+



In [116]:
# ✍️ Exercise:
# Display the DataFrame in full (no truncation e.g., truncate=False) and in vertical format
# write your code below:

df.show(vertical=True, truncate=False)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Flattening the Data Step by Step

The weather data is still nested, so it is not yet in a table format.

To understand how Spark transforms this data, we start with a simple step:

👉 flatten a single column.

We will use the `time` values inside the `daily` field to create one row per day.

In [117]:
# ✍️ Exercise:
# Use explode to flatten the column "daily.time"
# Create a new DataFrame with:
# - the city
# - one date per row
# Display the result
from pyspark.sql.functions import explode
# write your code below

df_time = df.select(
    "city",
    explode("daily.time").alias("date")
)

df_time.show()



+------+----------+
|  city|      date|
+------+----------+
|Madrid|2026-01-10|
|Madrid|2026-01-11|
|Madrid|2026-01-12|
|Madrid|2026-01-13|
|Madrid|2026-01-14|
|Madrid|2026-01-15|
|Madrid|2026-01-16|
|Madrid|2026-01-17|
|Madrid|2026-01-18|
|Madrid|2026-01-19|
|Madrid|2026-01-20|
|Madrid|2026-01-21|
|Madrid|2026-01-22|
|Madrid|2026-01-23|
|Madrid|2026-01-24|
|Madrid|2026-01-25|
|Madrid|2026-01-26|
|Madrid|2026-01-27|
|Madrid|2026-01-28|
|Madrid|2026-01-29|
+------+----------+
only showing top 20 rows


>👉 Each row now represents a single date.
>
>This is the first step in transforming nested data into a table.

## Understanding arrays_zip function

Before applying this to our weather data, let’s look at a simple example.

We will create a small DataFrame with two lists:
- dates
- temperatures

Then we will see how to combine them correctly.

In [118]:
from pyspark.sql import Row

example_data = [
    Row(
        city="ExampleCity",
        dates=["Day1", "Day2", "Day3"],
        temps=[20, 25, 22]
    )
]

df_example = spark.createDataFrame(example_data)
df_example.show(truncate=False)

df_example.printSchema()

+-----------+------------------+------------+
|city       |dates             |temps       |
+-----------+------------------+------------+
|ExampleCity|[Day1, Day2, Day3]|[20, 25, 22]|
+-----------+------------------+------------+

root
 |-- city: string (nullable = true)
 |-- dates: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- temps: array (nullable = true)
 |    |-- element: long (containsNull = true)



Let’s see what happens if we explode both columns directly.

In [119]:
from pyspark.sql.functions import explode

df_wrong = df_example.select(
    "city",
    explode("dates").alias("date"),
    explode("temps").alias("temp")
)

df_wrong.show()

+-----------+----+----+
|       city|date|temp|
+-----------+----+----+
|ExampleCity|Day1|  20|
|ExampleCity|Day1|  25|
|ExampleCity|Day1|  22|
|ExampleCity|Day2|  20|
|ExampleCity|Day2|  25|
|ExampleCity|Day2|  22|
|ExampleCity|Day3|  20|
|ExampleCity|Day3|  25|
|ExampleCity|Day3|  22|
+-----------+----+----+



> ⚠️ **Problem**
>
>The dates and temperatures are not matched correctly.
>
>👉 Spark creates all possible combinations instead of keeping values aligned  
>(this is similar to a cross join).

Now let’s fix this using `arrays_zip`.

In [120]:
from pyspark.sql.functions import arrays_zip

df_correct = df_example.select(
    "city",
    explode(arrays_zip("dates", "temps")).alias("data")
).select(
    "city",
    "data.dates",
    "data.temps"
)

df_correct.show()

+-----------+-----+-----+
|       city|dates|temps|
+-----------+-----+-----+
|ExampleCity| Day1|   20|
|ExampleCity| Day2|   25|
|ExampleCity| Day3|   22|
+-----------+-----+-----+



**Now it works correctly**

Each date is matched with the correct temperature.

### How `arrays_zip` + `explode` work together

#### 1. `arrays_zip(...)`

This function combines multiple lists **position by position**.

Example:

    dates = ["Day1", "Day2", "Day3"]
    temps = [20, 25, 22]

👉 After `arrays_zip`:

    [
      {dates: "Day1", temps: 20},
      {dates: "Day2", temps: 25},
      {dates: "Day3", temps: 22}
    ]

👉 Values are now correctly paired.

---

### 2. `explode(...)`

The `explode` function creates **one row per element** of a list.

So:

    explode(arrays_zip(...))

👉 becomes:

| date | temp |
|------|------|
| Day1 | 20   |
| Day2 | 25   |
| Day3 | 22   |

---

### Summary

- `arrays_zip` → keeps values aligned  
- `explode` → creates rows  

👉 Together, they transform nested lists into a clean table.

### Applying This to Our Data

Now that we understand how `arrays_zip` works, let’s apply it to our weather data.

Previously, we flattened only the dates.  
Now, we also want to keep the **maximum temperature for each day**.

👉 Each row should contain:
- the city  
- the date  
- the maximum temperature  

---

### ✍️ Exercise

Use `arrays_zip` and `explode` to:
- keep the values aligned  
- create one row per day

In [121]:
# ✍️ Exercise:
# Use arrays_zip and explode to create a new DataFrame with:
# - city
# - date
# - maximum temperature
# Display the result

# 💡 Hint:
# The columns you need are:
# - "daily.time"
# - "daily.temperature_2m_max"

from pyspark.sql.functions import arrays_zip, explode

# write your code below

df_two = df.select(
    "city",
    explode(
        arrays_zip("daily.time", "daily.temperature_2m_max")
    ).alias("weather")
).select(
    "city",
    "weather.time",
    "weather.temperature_2m_max"
)

df_two.show()

+------+----------+------------------+
|  city|      time|temperature_2m_max|
+------+----------+------------------+
|Madrid|2026-01-10|              NULL|
|Madrid|2026-01-11|              NULL|
|Madrid|2026-01-12|              NULL|
|Madrid|2026-01-13|              NULL|
|Madrid|2026-01-14|              NULL|
|Madrid|2026-01-15|              NULL|
|Madrid|2026-01-16|              NULL|
|Madrid|2026-01-17|              NULL|
|Madrid|2026-01-18|              NULL|
|Madrid|2026-01-19|              NULL|
|Madrid|2026-01-20|              NULL|
|Madrid|2026-01-21|              NULL|
|Madrid|2026-01-22|              NULL|
|Madrid|2026-01-23|              NULL|
|Madrid|2026-01-24|              NULL|
|Madrid|2026-01-25|              NULL|
|Madrid|2026-01-26|              NULL|
|Madrid|2026-01-27|              NULL|
|Madrid|2026-01-28|              NULL|
|Madrid|2026-01-29|              11.9|
+------+----------+------------------+
only showing top 20 rows


## 🌦️ Building the Full Weather Table

So far, we created a table with the **date** and the **maximum temperature**.

Now we will use the same idea to include more weather attributes for each day:

- date
- maximum temperature
- minimum temperature
- precipitation probability
- wind speed

👉 Each row should represent the weather for one city on one day.

In [122]:
# ✍️ Exercise:
# Create a DataFrame with one row per day including:
# - city
# - date
# - temperature_2m_max
# - temperature_2m_min
# - precipitation_probability_max
# - windspeed_10m_max
# Display the result

# 💡 Hint:
# Use arrays_zip with all the daily columns

from pyspark.sql.functions import arrays_zip, explode

# write your code below

df_flat = df.select(
    "city",
    explode(
        arrays_zip(
            "daily.time",
            "daily.temperature_2m_max",
            "daily.temperature_2m_min",
            "daily.precipitation_probability_max",
            "daily.windspeed_10m_max"
        )
    ).alias("weather")
).select(
    "city",
    "weather.time",
    "weather.temperature_2m_max",
    "weather.temperature_2m_min",
    "weather.precipitation_probability_max",
    "weather.windspeed_10m_max"
)

### Inspect the Result

Let’s inspect the DataFrame we just created.

In [123]:
# ✍️ Exercise:
# Show the schema and display the DataFrame

# write your code below
df_flat.printSchema()
df_flat.show()

root
 |-- city: string (nullable = true)
 |-- time: string (nullable = true)
 |-- temperature_2m_max: double (nullable = true)
 |-- temperature_2m_min: double (nullable = true)
 |-- precipitation_probability_max: long (nullable = true)
 |-- windspeed_10m_max: double (nullable = true)

+------+----------+------------------+------------------+-----------------------------+-----------------+
|  city|      time|temperature_2m_max|temperature_2m_min|precipitation_probability_max|windspeed_10m_max|
+------+----------+------------------+------------------+-----------------------------+-----------------+
|Madrid|2026-01-10|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-11|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-12|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-13|              NULL|              NULL|                     

# Working with Multiple Cities

So far, we have prepared weather data for a single city.

Now, we will repeat the same process for several cities and combine the results into one DataFrame.

👉 This will allow us to compare cities using the same weather criteria.

We will use the following list of cities:

In [124]:
cities = [
    ("Madrid", 40.41, -3.70),
    ("Paris", 48.85, 2.35),
    ("Berlin", 52.52, 13.41),
    ("Rome", 41.90, 12.49),
    ("London", 51.50, -0.12),
    ("Barcelona", 41.38, 2.17),
    ("Lisbon", 38.72, -9.14),
    ("Amsterdam", 52.37, 4.90),
    ("Brussels", 50.85, 4.35),
    ("Vienna", 48.21, 16.37),
    ("Prague", 50.08, 14.43),
    ("Copenhagen", 55.68, 12.57),
    ("Dublin", 53.35, -6.26),
    ("Athens", 37.98, 23.72),
    ("Zurich", 47.37, 8.54)
]

### Exercise

Create a single DataFrame containing weather data for all cities.

Steps:
1. Loop over the list of cities
2. Call `get_weather`
3. Load the result into Spark
4. Flatten the nested data
5. Combine all results into one DataFrame

👉 Reuse the code from the previous sections.

**Complete the `TODO` parts in the code below.**

In [125]:
from pyspark.sql.functions import arrays_zip, explode
import json

# We will store the DataFrame for each city in this list
dfs = []

# Loop over all cities in the list
# Each element contains: (city name, latitude, longitude)
for city, lat, lon in cities:

    # -----------------------------
    # Step 1: Get data from the API
    # -----------------------------
    # For each city, we call the function to retrieve weather data
    data = get_weather(city, lat, lon)

    # -----------------------------
    # Step 2: Load data into Spark
    # -----------------------------
    # Convert the JSON data into a Spark DataFrame
    df_temp = spark.read.json(
        spark.sparkContext.parallelize([json.dumps(data)])
    )

    # -----------------------------
    # Step 3: Flatten the data
    # -----------------------------
    # We transform the nested structure into a flat table
    # Each row will represent one day for the current city
    df_temp = df_temp.select(
        "city",
        explode(
            arrays_zip(
                # TODO: add the same columns as before
            )
        ).alias("weather")
    ).select(
        "city",
        # TODO: extract the fields from "weather"
    )

    # -----------------------------
    # Step 4: Store the result
    # -----------------------------
    # We add the DataFrame for this city to the list
    dfs.append(df_temp)

# -----------------------------
# Step 5: Combine all DataFrames
# -----------------------------
# At this point, we have one DataFrame per city in the list "dfs"

# Start with the first DataFrame
weather_df = dfs[0]

# Loop through the remaining DataFrames
# and combine them using union (stacking rows)
for d in dfs[1:]:
    weather_df = weather_df.union(d)

In [126]:
from pyspark.sql.functions import arrays_zip, explode
import json

# We will store the DataFrame for each city in this list
dfs = []

# Loop over all cities in the list
# Each element contains: (city name, latitude, longitude)
for city, lat, lon in cities:

    # -----------------------------
    # Step 1: Get data from the API
    # -----------------------------
    # For each city, we call the function to retrieve weather data
    data = get_weather(city, lat, lon)

    # -----------------------------
    # Step 2: Load data into Spark
    # -----------------------------
    # Convert the JSON data into a Spark DataFrame
    df_temp = spark.read.json(
        spark.sparkContext.parallelize([json.dumps(data)])
    )

    # -----------------------------
    # Step 3: Flatten the data
    # -----------------------------
    # We transform the nested structure into a flat table
    # Each row will represent one day for the current city
    df_temp = df_temp.select(
        "city",
        explode(
            arrays_zip(
                "daily.time",
                "daily.temperature_2m_max",
                "daily.temperature_2m_min",
                "daily.precipitation_probability_max",
                "daily.windspeed_10m_max"
            )
        ).alias("weather")
    ).select(
        "city",
        "weather.time",
        "weather.temperature_2m_max",
        "weather.temperature_2m_min",
        "weather.precipitation_probability_max",
        "weather.windspeed_10m_max"
    )

    # -----------------------------
    # Step 4: Store the result
    # -----------------------------
    # We add the DataFrame for this city to the list
    dfs.append(df_temp)

# -----------------------------
# Step 5: Combine all DataFrames
# -----------------------------
# At this point, we have one DataFrame per city in the list "dfs"

# Start with the first DataFrame
weather_df = dfs[0]

# Loop through the remaining DataFrames
# and combine them using union (stacking rows)
for d in dfs[1:]:
    weather_df = weather_df.union(d)

### Inspect the Result

Let’s inspect the DataFrame we just created.

In [127]:
# ✍️ Exercise:
# Show the first rows of the combined DataFrame

# write your code below
weather_df.show()

+------+----------+------------------+------------------+-----------------------------+-----------------+
|  city|      time|temperature_2m_max|temperature_2m_min|precipitation_probability_max|windspeed_10m_max|
+------+----------+------------------+------------------+-----------------------------+-----------------+
|Madrid|2026-01-10|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-11|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-12|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-13|              NULL|              NULL|                           90|             NULL|
|Madrid|2026-01-14|              NULL|              NULL|                           70|             NULL|
|Madrid|2026-01-15|              NULL|              NULL|                          100|             NULL|
|Madrid|2026-01-16|              NULL|        

👉 We now have a dataset with multiple cities.

Each row represents:
- one city
- one day
- its weather conditions

Next, we will use SQL to analyze and compare cities.

### Renaming Columns

The column names are long and not very easy to read.

Let’s rename them to make our queries simpler.

In [128]:
# ✍️ Exercise:
# Rename the columns to:
# - time -> date
# - temperature_2m_max -> temp_max
# - temperature_2m_min -> temp_min
# - precipitation_probability_max -> rain_prob
# - windspeed_10m_max -> wind_speed

# write your code below

weather_df = weather_df.withColumnRenamed("time", "date") \
    .withColumnRenamed("temperature_2m_max", "temp_max") \
    .withColumnRenamed("temperature_2m_min", "temp_min") \
    .withColumnRenamed("precipitation_probability_max", "rain_prob") \
    .withColumnRenamed("windspeed_10m_max", "wind_speed")

weather_df.printSchema()
weather_df.show()

root
 |-- city: string (nullable = true)
 |-- date: string (nullable = true)
 |-- temp_max: double (nullable = true)
 |-- temp_min: double (nullable = true)
 |-- rain_prob: long (nullable = true)
 |-- wind_speed: double (nullable = true)

+------+----------+--------+--------+---------+----------+
|  city|      date|temp_max|temp_min|rain_prob|wind_speed|
+------+----------+--------+--------+---------+----------+
|Madrid|2026-01-10|    NULL|    NULL|        0|      NULL|
|Madrid|2026-01-11|    NULL|    NULL|        0|      NULL|
|Madrid|2026-01-12|    NULL|    NULL|        0|      NULL|
|Madrid|2026-01-13|    NULL|    NULL|       90|      NULL|
|Madrid|2026-01-14|    NULL|    NULL|       70|      NULL|
|Madrid|2026-01-15|    NULL|    NULL|      100|      NULL|
|Madrid|2026-01-16|    NULL|    NULL|       82|      NULL|
|Madrid|2026-01-17|    NULL|    NULL|      100|      NULL|
|Madrid|2026-01-18|    NULL|    NULL|        1|      NULL|
|Madrid|2026-01-19|    NULL|    NULL|        0|      

👉 The table is now easier to read.

Each row represents:
- one city
- one day
- multiple weather measurements

We are now ready to analyze the data using SQL.

# Analyzing the Data with SQL

Now that we have a clean dataset with multiple cities, we can start analyzing it.

We will use **SQL in Spark** to answer questions about the weather.

👉 First, we register the DataFrame as a table.

In [129]:
weather_df.createOrReplaceTempView("weather")

### 🔎 Exploring the Data

Before writing more advanced queries, let’s explore the dataset.

👉 These quick queries will help us understand the size and structure of the data.

### 📊 Exercise 1 — How many rows?

Find the total number of rows in the dataset.

In [139]:
# ✍️ Exercise:
# Count the total number of rows in the table

# write your code below
spark.sql("""
SELECT COUNT(*) AS total_rows
FROM weather
""").show()

+----------+
|total_rows|
+----------+
|      1620|
+----------+



### 🏙️ Exercise 2 — How many cities?

Find how many different cities are in the dataset.

In [140]:
# ✍️ Exercise:
# Count the number of distinct cities

# write your code below
spark.sql("""
SELECT COUNT(DISTINCT city) AS num_cities
FROM weather
""").show()

+----------+
|num_cities|
+----------+
|        15|
+----------+



### 📅 Exercise 3 — How many months?

Find how many different months are present in the dataset.

In [141]:
# ✍️ Exercise:
# Count the number of distinct months in the data

# write your code below
spark.sql("""
SELECT COUNT(DISTINCT MONTH(to_date(date))) AS num_months
FROM weather
""").show()

+----------+
|num_months|
+----------+
|         4|
+----------+



### 📆 Exercise 4 — Which months are included?

List the months present in the dataset.

In [144]:
# ✍️ Exercise:
# list the months included

# write your code below
spark.sql("""
SELECT DISTINCT MONTH(to_date(date)) AS month
FROM weather
ORDER BY month
""").show()

+-----+
|month|
+-----+
|    1|
|    2|
|    3|
|    4|
+-----+



### ☀️ Challenge 1 — Find Warm and Dry Days

Find all days where:
- the maximum temperature is at least 20°C  
- the probability of rain is less than 20%  

👉 Display the result.

In [130]:
# ✍️ Exercise:
# Write a SQL query to find warm and dry days

# write your code below
spark.sql("""
SELECT *
FROM weather
WHERE temp_max >= 20
AND rain_prob < 20
""").show()

+------+----------+--------+--------+---------+----------+
|  city|      date|temp_max|temp_min|rain_prob|wind_speed|
+------+----------+--------+--------+---------+----------+
|Madrid|2026-02-23|    21.5|     2.9|        0|       6.6|
|Madrid|2026-02-24|    20.7|     4.5|        0|       9.3|
|Madrid|2026-02-25|    20.6|     5.2|        0|       5.9|
|Madrid|2026-02-26|    21.0|     5.9|        0|       5.9|
|Madrid|2026-03-17|    21.9|     4.8|        0|       7.1|
|Madrid|2026-03-25|    21.6|     5.1|        0|      16.1|
|Madrid|2026-04-03|    21.4|     4.8|        0|      14.0|
|Madrid|2026-04-04|    23.8|     7.3|        0|       6.3|
|Madrid|2026-04-05|    25.4|     9.4|        0|       5.8|
|Madrid|2026-04-06|    25.3|    11.0|        0|      10.7|
|Madrid|2026-04-09|    25.6|     8.3|        0|       8.9|
|Madrid|2026-04-10|    27.1|    14.1|        5|       9.0|
|Madrid|2026-04-14|    20.1|     7.8|        0|      13.9|
|Madrid|2026-04-15|    23.5|     8.6|        0|       6.

### 🏆 Challenge 2 — Which City Has the Most Nice Days?

Using the same definition as before (warm and dry):

👉 Count how many "nice days" each city has  
👉 Sort the results from highest to lowest

In [131]:
# ✍️ Exercise:
# Write a SQL query to rank cities by number of nice days

# write your code below
spark.sql("""
SELECT
    city,
    COUNT(*) AS nice_days
FROM weather
WHERE temp_max >= 20
AND rain_prob < 20
GROUP BY city
ORDER BY nice_days DESC
""").show()

+---------+---------+
|     city|nice_days|
+---------+---------+
|   Lisbon|       28|
|   Madrid|       20|
|     Rome|       17|
|   Athens|       15|
|Barcelona|        9|
|    Paris|        7|
|   Zurich|        5|
| Brussels|        4|
|Amsterdam|        3|
|   Berlin|        2|
|   Vienna|        2|
|   London|        1|
|   Prague|        1|
+---------+---------+



### 🪁 Challenge 3 — Windy Days

Find the cities with the most windy days.

A day is considered **windy** if:
- wind speed is between 15 and 30  
- probability of rain is less than 20%  

👉 Count how many windy days each city has and rank them.

In [132]:
# ✍️ Exercise:
# Write a SQL query to find and rank cities by number of windy days

# write your code below
spark.sql("""
SELECT
    city,
    COUNT(*) AS windy_days
FROM weather
WHERE wind_speed BETWEEN 15 AND 30
AND rain_prob < 20
GROUP BY city
ORDER BY windy_days DESC
""").show()

+----------+----------+
|      city|windy_days|
+----------+----------+
|    Lisbon|        29|
| Barcelona|        26|
| Amsterdam|        19|
|    Vienna|        19|
|Copenhagen|        18|
|    Athens|        18|
|    Madrid|        16|
|    London|        16|
|     Paris|        13|
|    Berlin|        12|
|  Brussels|        12|
|    Prague|         9|
|      Rome|         8|
|    Dublin|         6|
|    Zurich|         4|
+----------+----------+



### 🎯 Challenge 4 — Your perfect weather

Now it's your turn!

👉 Define your own "perfect weather" using:
- temperature
- rain probability
- wind speed

Then:
- find matching days
- rank the cities

Be creative!

In [133]:
# ✍️ Exercise:
# Define your own perfect weather and rank the cities

# write your code below
spark.sql("""
SELECT
    city,
    COUNT(*) AS perfect_days
FROM weather
WHERE temp_max BETWEEN 18 AND 24
AND wind_speed BETWEEN 5 AND 20
AND rain_prob < 20
GROUP BY city
ORDER BY perfect_days DESC
""").show()

+---------+------------+
|     city|perfect_days|
+---------+------------+
|   Athens|          24|
|   Madrid|          22|
|     Rome|          22|
|Barcelona|          22|
|   Lisbon|          19|
|    Paris|          17|
| Brussels|          13|
|   Berlin|           5|
|   London|           5|
|   Vienna|           5|
|Amsterdam|           4|
|   Prague|           3|
|   Zurich|           3|
+---------+------------+



### 📅 Challenge 5 — Best Month to Visit

So far, we have analyzed weather day by day.

But in real life, people plan trips by **month**, not by individual days.

👉 Let’s find out:

**Which is the best month to visit each city?**

### Your task

1. Convert the `date` column to a proper date format  
2. Extract the year and month  
3. Calculate the **average maximum temperature per city per month**  
4. Sort the results to identify the warmest months  


👉 Use the following functions:
- `to_date()`
- `month()`
- `year()`

In [134]:
# ✍️ Exercise:
# Analyze the average maximum temperature per city per month
from pyspark.sql.functions import to_date, month, year, avg
# write your code below


df_date = weather_df.withColumn("date", to_date("date"))

result = df_date.groupBy(
    "city",
    year("date").alias("year"),
    month("date").alias("month")
).agg(
    avg("temp_max").alias("avg_temp_max")
).orderBy("city", "month")

result.show()

+---------+----+-----+------------------+
|     city|year|month|      avg_temp_max|
+---------+----+-----+------------------+
|Amsterdam|2026|    1| 3.733333333333333|
|Amsterdam|2026|    2| 8.139285714285714|
|Amsterdam|2026|    3|12.793548387096774|
|Amsterdam|2026|    4|15.733333333333334|
|   Athens|2026|    1|16.166666666666668|
|   Athens|2026|    2|16.121428571428574|
|   Athens|2026|    3|16.661290322580644|
|   Athens|2026|    4| 20.47037037037037|
|Barcelona|2026|    1|15.166666666666666|
|Barcelona|2026|    2|16.124999999999996|
|Barcelona|2026|    3|16.180645161290318|
|Barcelona|2026|    4| 19.04074074074074|
|   Berlin|2026|    1|              -1.0|
|   Berlin|2026|    2|              4.15|
|   Berlin|2026|    3|13.074193548387095|
|   Berlin|2026|    4|13.844444444444443|
| Brussels|2026|    1| 7.033333333333334|
| Brussels|2026|    2|10.453571428571427|
| Brussels|2026|    3|14.493548387096778|
| Brussels|2026|    4|17.637037037037036|
+---------+----+-----+------------

### 📊 Bonus Challenge — Which City Has the Most Consistent Weather?

A city can be pleasant not only because it is warm, but also because its weather is stable.

👉 We will use a new function: `STDDEV_POP`

### What does it do?

- It measures how much values vary from the average  
- A **low value** means the temperature is stable (consistent)  
- A **high value** means the temperature changes a lot  

👉 In our case:
- low variation = more predictable weather  

### Your task

Find the cities with the most consistent maximum temperature.


In [136]:
# ✍️ Bonus Exercise:
# Find the cities with the most consistent maximum temperature

# write your code below

spark.sql("""
SELECT
    city,
    ROUND(AVG(temp_max), 2) AS avg_temp_max,
    ROUND(STDDEV_POP(temp_max), 2) AS temp_variation
FROM weather
GROUP BY city
ORDER BY temp_variation ASC
""").show()

+----------+------------+--------------+
|      city|avg_temp_max|temp_variation|
+----------+------------+--------------+
| Barcelona|        17.0|          2.51|
|    Dublin|       11.28|          2.55|
|    Athens|       17.63|          2.71|
|      Rome|       17.66|          3.21|
|    Lisbon|       18.41|          3.39|
|    London|       13.07|          3.54|
|     Paris|       14.75|          4.04|
| Amsterdam|       11.92|          4.58|
|  Brussels|       13.92|          4.71|
|    Zurich|       11.98|          4.98|
|    Madrid|       16.78|          4.99|
|Copenhagen|        7.25|          5.13|
|    Vienna|       11.65|          5.47|
|    Prague|       10.56|          5.57|
|    Berlin|       10.03|          6.54|
+----------+------------+--------------+

